# COGS 4290 Assignment 0

## 📥 Saving your answers for grading

This assignment is **auto-graded**. After each question there is a **grader cell**
that saves the data you plotted or computed into an `answers/Module_01/` folder so
it can be compared against the reference answers.

**For each question:**
1. The question tells you which result(s) to produce and the expected data
   structure/format. Do your analysis and bind each result to a variable.
2. In the grader cell, replace the placeholder variable with **your** variable name.
3. Run the grader cell — it calls `save_answer(...)` and writes your answer.

Your **plots are saved too** — make sure each plotting cell calls `plt.show()` so the
figure can be captured for the grade report.

Make sure every grader cell runs without error before you submit. You don't need to
change anything else.


In [ ]:
# grader setup — enables saving the figures you plot (run once, early)
try:
    from grader.answer_io import enable_figure_capture
    enable_figure_capture()
except Exception as _e:
    print("grader: figure capture not enabled:", _e)


A quick note: Not everything needed to complete this assignment is covered in the previous introduction. You will have to do some individual researching at some points; however, everything you need is accessible with a quick google. Best of luck! 

In [1]:
# imports
import numpy as np
import pandas as pd; pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import pickle
import time

## Problem 1: Numpy and Multi-dimensional Data

<!-- grader-note -->
> **📥 For grading, produce and save:** `day_cold` (scalar — day-of-year index (0-364) with the coldest whole-lake mean temperature) → Q1.1; `day_warm` (scalar — day-of-year index (0-364) with the warmest whole-lake mean temperature) → Q1.1; `layer_stdevs` (array (5,) — sample std dev (ddof=1) of the whole-lake daily-mean temperature for each 10 m depth layer, ordered 0-10, 10-20, 20-30, 30-40, 40-50 m) → Q1.2; `t_summer_week` (array (50, 100) — 2-D depth x length temperature profile averaged over the summer week (averaged over width and the 7 days)) → Q1.3; `t_winter_week` (array (50, 100) — 2-D depth x length temperature profile averaged over the winter week (averaged over width and the 7 days)) → Q1.3. Bind each to a variable, then run the grader cell(s) below.
<!-- /grader-note -->

We have data on the temperature profile of a lake in Switzerland, in the form of a 4-dimensional array.  The first dimension represents depth, the second length, the third width (all in meters), and the last dimension represents the day of the year (stating January 1st and ending December 31st). We are interested in the dynamics of the temperature profile of the lake over the course of the year and want to answer the follwing.

Questions:
1) Find the day of the year where the average temperature of the entire lake is the coldest, warmest.
2) For each 10 meter layer of depth, find the sample standard deviation of temperatue over the year, treating each day as independent.
3) Averaging over the width, plot the 2-dimensional temperatue profile of the lake averaging over 1 week in the summer and 1 week in the winter. **(Hint: You are looking for an average temperature value at each length and width)**

Running the cell below will generate the data for use to analyze.

In [2]:
def gen_lake_data():
    temp_lake = np.zeros((50, 100, 70, 365), dtype=float)    # 50 meters depth, 100 meters length, 50 meters width, 365 days
    ones = np.ones((temp_lake.shape[1], temp_lake.shape[2]))
    
    # linear function of height
    h = np.arange(0, 50)
    temp_h = -0.3*h + 15
    
    # sinusoid on time
    day = np.arange(0, 365)
    temp_day = np.sin(np.radians(day) - 5*np.pi/6)
    
    for i in range(temp_lake.shape[0]):
        for j in range(temp_lake.shape[3]):
            np.random.seed(i*j)
            temp_lake[i,:,:,j] = temp_h[i] * temp_day[j] * ones * 0.8 + np.random.uniform(-1.0, 1.0, ones.shape) + 10   # randomly sample for each width and length
    
    return temp_lake

t_lake = gen_lake_data()

#### Part (a)
Find the day of the year where the average temperature of the entire lake is the coldest and the warmest.

In [3]:
### YOUR CODE HERE


In [4]:
# print results


#### Part (b)
For each 10 meter layer of depth, find the sample standard deviation of temperature over the year, treating each day as independent.  First, do so manually using the formula for the sample standard deviation:

$$
\sigma = \sqrt{\frac{\sum{(x_{i}-\bar{x})^{2}}}{n-1}}
$$

where we sum over all observations $x_{i}$, $\bar{x}$ is the sample mean, and $n$ is the number of data points.  Then, do the same calculation using a numpy function.

In [5]:
### YOUR CODE HERE


In [6]:
# manually calculate


In [7]:
# print results


In [8]:
# use a numpy function


In [9]:
# print results


#### Part (c)
Plot the temperature profile averaged over 1 week in the summer and 1 week in the winter. 

In [10]:
### YOUR CODE HERE


In [11]:
# graphing


In [ ]:
# ── grader cell (Question 1.1) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q1.1_day_cold", day_cold, module=1, question="1.1")   # ← replace `day_cold` with your variable
save_answer("Q1.1_day_warm", day_warm, module=1, question="1.1")   # ← replace `day_warm` with your variable

In [ ]:
# ── grader cell (Question 1.2) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q1.2_layer_stdevs", layer_stdevs, module=1, question="1.2")   # ← replace `layer_stdevs` with your variable

In [ ]:
# ── grader cell (Question 1.3) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q1.3_t_summer_week", t_summer_week, module=1, question="1.3", fig="last")   # ← replace `t_summer_week` with your variable
save_answer("Q1.3_t_winter_week", t_winter_week, module=1, question="1.3")   # ← replace `t_winter_week` with your variable

## Problem 2: Pandas

<!-- grader-note -->
> **📥 For grading, produce and save:** `qualifying_per_league` (json — number of qualifying midfielders (played >10 90s) per league (Comp), as a league->count mapping) → Q2.2; `mean_cmp_pct` (scalar — mean total pass completion % across qualifying midfielders) → Q2.3; `mean_kp_per90` (scalar — mean key passes per 90 across qualifying midfielders) → Q2.3; `mean_prgdist_per90` (scalar — mean progressive passing distance per 90 across qualifying midfielders) → Q2.3; `filtered_players` (labels — player names above the mean on all three criteria (completion %, KP/90, PrgDist/90)) → Q2.3; `players_to_sign` (labels — final list of players to sign after adding the under-25 youth filter) → Q2.4; `se_cmp_pct` (scalar — standard error of the mean completion % for the players-to-sign subset) → Q2.4; `se_prgdist_per90` (scalar — standard error of the mean progressive-distance/90 for the players-to-sign subset) → Q2.4; `se_kp_per90` (scalar — standard error of the mean key-passes/90 for the players-to-sign subset (solution reuses the name kp_90 for this SE)) → Q2.4. Bind each to a variable, then run the grader cell(s) below.
<!-- /grader-note -->

Imagine you work for a professional soccer club.  Your team just lost your best player, so the owner has tasked you with identifying potential replacement players that your team can sign.  Specifically, the manager has instructed you that they are looking for players with the following qualifications:

1) Midfielders: Only look at players who have 'MF' listed as a position they can play ('Pos').
2) Injury-Free: Only look at players who played more that 10 90s in the previous season ('90s').
3) Above average passing completion percentage ('Total_CMP%').
4) Above average key passes per 90 minutes ('KP').
5) Above average progressive passing distance per 90 minutes ('Total_PrgDist').
6) Youth: Only look at players under 25 years old ('Age').

Run the following code to load in a dataset containing player statistics from the previous season.

In [12]:
def load_fbref_data():
    # load in data from html
    passing_t5 = pd.read_html('https://fbref.com/en/comps/Big5/2022-2023/passing/players/2022-2023-Big-5-European-Leagues-Stats')[0]
    # relabel column names
    passing_t5.columns = [x[0] + '_' + x[1] if 'Unnamed' not in x[0] else x[1] for x in passing_t5.columns]
    # clean up ages
    passing_t5['Age'] = [x.split('-')[0] if type(x)==str else '100' for x in passing_t5.Age]
    # convert from strings to numerics
    num_cols = ['Age', 'Born', '90s', 'Total_Cmp', 'Total_Att', 'Total_Cmp%', 'Total_TotDist', 'Total_PrgDist', 'Short_Cmp', 'Short_Att', 'Short_Cmp%', 
                'Medium_Cmp', 'Medium_Att', 'Medium_Cmp%', 'Long_Cmp', 'Long_Att', 'Long_Cmp%', 'Ast', 'xAG', 'xA', 'A-xAg', 'KP', '1/3', 'PPA', 'CrsPA', 'PrgP']
    for c in passing_t5.columns:
        if c in num_cols:
            passing_t5[c] = pd.to_numeric(passing_t5[c], errors='coerce')
    
    
    return passing_t5

dataframe = load_fbref_data()
dataframe

HTTPError: HTTP Error 403: Forbidden

#### Part (a)
First, let's clean up our dataset a little bit.  There are numerous columns we are not concerned with, so let's just remove them.  Drop all of the columns in the list below from our dataframe.

In [ ]:
cols_to_drop = ['Rk', 'Born', 'xAG', 'A-xAG', '1/3', 'PPA', 'CrsPA', 'Matches']


### YOUR CODE HERE


#### Part (b)
Now, let's begin filtering our data.  First, let's determine how many players qualify based on our first 2 criteria.  Filter out all players who are not midfielders or who played fewer than 10 90s last season.  Report the number of players who qualify from each league ('Comp').

In [ ]:
### YOUR CODE HERE


In [ ]:
# print results


#### Part (c)
For the next 3 criteria, we want to judge individual performance in relation to the performance of other players.  Also, we want to look at statistics based on a per 90 minute rate. Using the data we have available, create two new columns ('KP_per90' and 'Total_PrgDist_per90') and populate them with the rate data we want. Then, filter the data further and identify the list of players who had above average passing completion percentage, key passes per 90 minutes, and progressive passing distance per 90 minutes.  Also report the means for these three statistics.

In [ ]:
### YOUR CODE HERE


In [25]:
# print results


#### Part (d)
Finally, filter based on the final youth criteria, and present a list of players who the team should try to sign.  Report the standard error of the mean of our 3 criteria from part (c) for this subset of players.  Recall the formula for the standard error:

$$
SE = \frac{\sigma}{\sqrt{n}}
$$

where $\sigma$ is the sample standard deviation, and $n$ is the number of independent observations.

In [26]:
### YOUR CODE HERE


In [27]:
# print results


In [ ]:
# ── grader cell (Question 2.2) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q2.2_qualifying_per_league", qualifying_per_league, module=1, question="2.2")   # ← replace `qualifying_per_league` with your variable

In [ ]:
# ── grader cell (Question 2.3) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q2.3_mean_cmp_pct", mean_cmp_pct, module=1, question="2.3")   # ← replace `mean_cmp_pct` with your variable
save_answer("Q2.3_mean_kp_per90", mean_kp_per90, module=1, question="2.3")   # ← replace `mean_kp_per90` with your variable
save_answer("Q2.3_mean_prgdist_per90", mean_prgdist_per90, module=1, question="2.3")   # ← replace `mean_prgdist_per90` with your variable
save_answer("Q2.3_filtered_players", filtered_players, module=1, question="2.3")   # ← replace `filtered_players` with your variable

In [ ]:
# ── grader cell (Question 2.4) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q2.4_players_to_sign", players_to_sign, module=1, question="2.4")   # ← replace `players_to_sign` with your variable
save_answer("Q2.4_se_cmp_pct", se_cmp_pct, module=1, question="2.4")   # ← replace `se_cmp_pct` with your variable
save_answer("Q2.4_se_prgdist_per90", se_prgdist_per90, module=1, question="2.4")   # ← replace `se_prgdist_per90` with your variable
save_answer("Q2.4_se_kp_per90", se_kp_per90, module=1, question="2.4")   # ← replace `se_kp_per90` with your variable

## Problem 3: Reading, Parsing, and Writing Files
The opening few pages to Edward Albee's play "A Delicate Balance" are stored in the text file sample_files/Albee_A_Delicate_Balance.txt.  Each line in the text file contains one line of dialogue, with the start of the line indicating the character speaking (i.e. CHARACTER_NAME: line of dialogue).

For each question, you will append your answer to the "answers" list.

<!-- grader-note -->
> **📥 For grading, produce and save:** `char_line_counts` (json — number of lines each character speaks, as a character->count mapping (prompt orders them by first appearance; order is not auto-graded)) → Q3.1; `julia_count` (scalar — number of times the substring 'Julia' is said across all lines) → Q3.2; `line_10th_words` (json — per line (in order): the 10th alphabetic word, or the line's alphabetic-word count if the line has fewer than 10 words) → Q3.3. Bind each to a variable, then run the grader cell(s) below.
<!-- /grader-note -->

In [28]:
answers = []

#### Part (a)
Determine how many lines each character has.  Store in the answers list as (character, number of lines) tuples, which should be ordered based on the order in which the characters' first lines appear in the play.

In [29]:
### YOUR CODE HERE


In [30]:
# append to answers list


#### Part (b)
Determine how many times a character says "Julia."

In [31]:
### YOUR CODE HERE


In [32]:
# append to answers list


#### Part (c)
Find the 10th word of every line and add it to your list of answers.  For lines with fewer than 10 words, instead add the number of words on the line to your list of answers.  Note that the character name does not count as a word, and that we only want words (not numbers or special characters).

In [33]:
### YOUR CODE HERE


In [34]:
# append to answers list


In [ ]:
# ── grader cell (Question 3.1) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q3.1_char_line_counts", char_line_counts, module=1, question="3.1")   # ← replace `char_line_counts` with your variable

In [ ]:
# ── grader cell (Question 3.2) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q3.2_julia_count", julia_count, module=1, question="3.2")   # ← replace `julia_count` with your variable

In [ ]:
# ── grader cell (Question 3.3) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q3.3_line_10th_words", line_10th_words, module=1, question="3.3")   # ← replace `line_10th_words` with your variable

## Problem 4: Randomize
Randomize a list of $n$ items.  First, do so with only using *np.random.randint* to generate 1 random integer at a time.  Then, do so with *np.random.shuffle*.  Compare how long the two methods take for lists of length 1000, and 1000000.
* Hints: *time.time()* will get the current time, which we can use to calculate how long code takes to run.

In [36]:
# lists of different lengths
lst_thousand = np.arange(1000)
lst_million = np.arange(1000000)

In [37]:
def randomize(lst):
    ### YOUR CODE HERE
    raise NotImplementedError
    
def shuffle(lst):
    new_lst = lst.copy()
    np.random.shuffle(new_lst)
    return new_lst

In [38]:
### YOUR CODE HERE


In [39]:
# print results
